# Training

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

# ML 
import torch
import numpy as np

# Modules
import sys
sys.path.append("../")

from inference import create_model_architecture
from io_utils import add_results
from core import prepare_data, create_learner_object

# Further 
from fastai.callback.all import SaveModelCallback

# import csv
from pathlib import Path 

import joblib


## Functions for the Separate Training

In [ ]:
def train_central(run, partition, models_path, model_name, epochs, lr = 0.001, csv_path = "./results.csv", s=2, h=1, c_in=87, seq_len=96):

    # Find GPU, create DLS 
    device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
    dls, _, means, stds = prepare_data(partition, s=s, h=h)

    # Check and Create Path
    target_path = Path(models_path).resolve()
    target_path.mkdir(parents=True, exist_ok=True)

    # Safe means and stds
    stats_file = target_path / 'preprocessing_stats.joblib'

    if not stats_file.exists():
        target_path.mkdir(parents=True, exist_ok=True)
        joblib.dump({'means': means, 'stds': stds}, stats_file)

    # Create model and learer 
    model, model_name = create_model_architecture(model_name,  c_in=c_in, c_out=h, seq_len=seq_len)
    learn = create_learner_object(dls, model, device)

    # Training
    learn.path = target_path
    learn.fit_one_cycle(epochs, lr, cbs=SaveModelCallback(fname=model_name))
    learn.load(model_name)

    # Determine best result
    best_idx = np.argmin([r[1] for r in learn.recorder.values])
    val = learn.recorder.values[best_idx]
    
    # Safe as an array 
    results = {
        "train_loss": float(val[0]),     
        "val_loss": float(val[1]), 
        "mse": float(val[2]),
        "mae": float(val[3]),
        "rmse": float(val[4])
    }

    # Set Mode and add results to the csv file
    mode="central"
    mapping = {0: 'a', 1: 'b', 2: 'c'}
    mapped_val = mapping[partition]

    add_results(csv_path, run, mode, model_name, epochs, mapped_val, results)
    
    return results

## Separate Training

In [ ]:
models = [
    ("LSTM", 30),
    ("gMLP", 15),
    ("InceptionTime", 15),
    ("TSTPlus", 20),
    ("GRUPlus", 20),
]


# for run in range(3, 7):
for run in [10]:

    for partition, letter in [(0, "a"), (1, "b"), (2, "c")]:

        dynamic_path = f"../models/central/run_{run}/{letter}"

        for model, epochs in models:
            train_central(
                run=run, 
                partition=partition, 
                models_path=dynamic_path,
                model_name=model, 
                epochs=epochs, 
                h=30
            )

epoch,train_loss,valid_loss,mse,mae,_rmse,time
0,1042.859619,1050.626465,1050.626465,28.287756,32.413368,00:02
1,1036.675171,1037.957275,1037.957275,28.066444,32.217342,00:02
2,898.755798,861.884521,861.884521,24.618637,29.357872,00:02
3,670.697205,652.257690,652.257690,19.898428,25.539337,00:01
4,484.801208,444.222015,444.222015,14.102824,21.076576,00:02
5,363.662109,341.391541,341.391541,11.045897,18.476786,00:02
6,308.721771,296.452759,296.452759,10.437974,17.217804,00:02
7,292.371735,283.565674,283.565674,10.639184,16.839407,00:02
8,282.305969,278.428253,278.428253,10.240319,16.686167,00:02
9,253.031128,272.179932,272.179932,10.156534,16.497875,00:01


Better model found at epoch 0 with valid_loss value: 1050.62646484375.
Better model found at epoch 1 with valid_loss value: 1037.957275390625.
Better model found at epoch 2 with valid_loss value: 861.884521484375.
Better model found at epoch 3 with valid_loss value: 652.2576904296875.
Better model found at epoch 4 with valid_loss value: 444.2220153808594.
Better model found at epoch 5 with valid_loss value: 341.39154052734375.
Better model found at epoch 6 with valid_loss value: 296.4527587890625.
Better model found at epoch 7 with valid_loss value: 283.565673828125.
Better model found at epoch 8 with valid_loss value: 278.4282531738281.
Better model found at epoch 9 with valid_loss value: 272.179931640625.
Better model found at epoch 14 with valid_loss value: 271.75567626953125.
Better model found at epoch 21 with valid_loss value: 265.7578430175781.
Better model found at epoch 24 with valid_loss value: 259.77972412109375.


epoch,train_loss,valid_loss,mse,mae,_rmse,time
0,494.634186,443.612823,443.612823,16.764896,21.062117,00:14
1,237.471985,213.549713,213.549713,8.671504,14.613339,00:14
2,273.092651,213.786560,213.786560,9.563347,14.621442,00:14
3,245.995422,326.681183,326.681183,10.461538,18.074324,00:14
4,155.431839,215.415024,215.415024,10.841467,14.677024,00:14
5,174.933502,198.647812,198.647812,8.986956,14.094248,00:14
6,129.396225,183.625244,183.625244,10.035748,13.550838,00:17
7,110.573296,180.931168,180.931168,9.737707,13.451066,00:19
8,103.021889,171.612015,171.612015,10.245929,13.100077,00:21
9,83.845558,138.545197,138.545197,8.697317,11.770522,00:21


Better model found at epoch 0 with valid_loss value: 443.6128234863281.
Better model found at epoch 1 with valid_loss value: 213.54971313476562.
Better model found at epoch 5 with valid_loss value: 198.64781188964844.
Better model found at epoch 6 with valid_loss value: 183.625244140625.
Better model found at epoch 7 with valid_loss value: 180.93116760253906.
Better model found at epoch 8 with valid_loss value: 171.6120147705078.
Better model found at epoch 9 with valid_loss value: 138.54519653320312.
Better model found at epoch 10 with valid_loss value: 108.8299331665039.
Better model found at epoch 11 with valid_loss value: 93.48787689208984.


epoch,train_loss,valid_loss,mse,mae,_rmse,time
0,1009.981262,757.619446,757.619446,22.876873,27.524887,00:09
1,849.834351,598.774658,598.774658,20.109781,24.469873,00:09
2,500.245331,327.355652,327.355652,14.154910,18.092972,00:08
3,294.486420,326.372131,326.372131,13.232276,18.065773,00:08
4,236.313736,287.408478,287.408478,12.153098,16.953125,00:07
5,193.625565,258.210663,258.210663,10.399910,16.068935,00:07
6,167.104126,297.844269,297.844269,11.936746,17.258165,00:07
7,145.038666,320.020752,320.020752,12.259270,17.889124,00:07
8,129.690262,338.553284,338.553284,12.634218,18.399817,00:08
9,117.479942,333.851868,333.851868,13.072764,18.271616,00:08


...

epoch,train_loss,valid_loss,mse,mae,_rmse,time
0,397.202576,927.725403,927.725403,20.364193,30.458586,00:11
1,126.884354,316.464325,316.464325,12.629248,17.789446,00:13
2,93.502205,235.288925,235.288925,12.744050,15.339130,00:12
3,90.321129,228.984970,228.984970,10.427385,15.132247,00:11
4,85.339706,144.898407,144.898407,9.079381,12.037374,00:11
5,83.385780,211.930359,211.930359,11.887304,14.557828,00:12
6,77.325150,166.537781,166.537781,10.103597,12.904950,00:12
7,67.750984,1578.453247,1578.453247,16.419170,39.729748,00:13
8,67.304070,210.386719,210.386719,11.936667,14.504714,00:14
9,65.026276,239.436340,239.436340,12.937296,15.473730,00:14


Better model found at epoch 0 with valid_loss value: 927.7254028320312.
Better model found at epoch 1 with valid_loss value: 316.4643249511719.
Better model found at epoch 2 with valid_loss value: 235.28892517089844.
Better model found at epoch 3 with valid_loss value: 228.98497009277344.
Better model found at epoch 4 with valid_loss value: 144.89840698242188.
Better model found at epoch 12 with valid_loss value: 132.29627990722656.
Better model found at epoch 14 with valid_loss value: 102.88053894042969.


epoch,train_loss,valid_loss,mse,mae,_rmse,time
0,824.652588,936.733459,936.733459,28.468517,30.606100,00:13
1,806.426331,848.555176,848.555176,26.894482,29.129971,00:13
2,561.315186,515.045349,515.045349,19.789772,22.694611,00:13
3,306.177094,270.576080,270.576080,13.594593,16.449198,00:13
4,168.832016,160.048401,160.048401,9.931418,12.651024,00:13
5,114.061874,116.700768,116.700768,7.861761,10.802813,00:13
6,91.264076,104.609253,104.609253,7.284184,10.227866,00:13
7,85.113121,96.252541,96.252541,6.978809,9.810839,00:13
8,74.279984,94.910591,94.910591,6.948824,9.742207,00:13
9,70.705566,90.489174,90.489174,6.780150,9.512580,00:13


Better model found at epoch 0 with valid_loss value: 936.7334594726562.
Better model found at epoch 1 with valid_loss value: 848.55517578125.
Better model found at epoch 2 with valid_loss value: 515.0453491210938.
Better model found at epoch 3 with valid_loss value: 270.5760803222656.
Better model found at epoch 4 with valid_loss value: 160.04840087890625.
Better model found at epoch 5 with valid_loss value: 116.70076751708984.
Better model found at epoch 6 with valid_loss value: 104.6092529296875.
Better model found at epoch 7 with valid_loss value: 96.2525405883789.
Better model found at epoch 8 with valid_loss value: 94.91059112548828.
Better model found at epoch 9 with valid_loss value: 90.48917388916016.
Better model found at epoch 10 with valid_loss value: 84.58910369873047.
Better model found at epoch 11 with valid_loss value: 84.46929931640625.
Better model found at epoch 12 with valid_loss value: 80.52359008789062.
Better model found at epoch 13 with valid_loss value: 77.22803

: 